# Calligraphy Dataset Preprocessing to 96x96 PNG

This notebook has one purpose: preprocess all target calligraphy single-character images from the six local bundles and write 96x96 grayscale PNG files.

It does not train models, split data, evaluate classifiers, or run sampling analysis.


## 1. Imports and configuration

All input and output paths stay inside `/Users/lainantian/Developer/Caligraphy`.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import math
import re
import unicodedata
import traceback

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

try:
    from scipy import ndimage as ndi
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

ROOT = Path('/Users/lainantian/Developer/Caligraphy')
OUTPUT_ROOT = ROOT / 'Caligraphy_preprocessed_96'
OUTPUT_IMAGE_ROOT = OUTPUT_ROOT / 'images'

RUN_FULL = True
SAMPLE_SIZE = 100  # Debug only. Formal run uses all rows because RUN_FULL=True.
RANDOM_SEED = 42

IMAGE_SIZE = 96
MIN_AREA = 30

BUNDLE_NAMES = [
    'common_c_bundle',
    'common_g_bundle 2',
    'common_h_bundle',
    'common_i_bundle',
    'common_j_bundle',
    'less_common_b_bundle',
]

ORIGINAL_COLUMNS = [
    'query_char', 'style_code', 'style_name', 'era', 'author',
    'work_title', 'detail_title', 'image_url', 'thumb_url'
]

STYLE_MAP = {
    '楷书': '楷書',
    '篆书': '篆書',
    '草书': '草書',
    '章草': '草書',
    '行书': '行書',
    '隶书': '隸書',
}
STYLE_LABEL_ORDER = ['楷書', '篆書', '草書', '行書', '隸書']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

VECTOR_HINT_PHRASES = ['如果不满意这么大', '矢量部分', '找到原大的']
TEXT_COLUMNS = ORIGINAL_COLUMNS

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
for style_label in STYLE_LABEL_ORDER:
    (OUTPUT_IMAGE_ROOT / style_label).mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'OUTPUT_ROOT: {OUTPUT_ROOT}')
print(f'RUN_FULL: {RUN_FULL}')
print(f'OpenCV available: {HAS_CV2}; SciPy available: {HAS_SCIPY}')


## 2. Text, filename, and vector-hint helpers

These helpers normalize text for matching, sanitize output filenames, and skip website vector-hint images without OCR.


In [ ]:
def norm_text(value) -> str:
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except TypeError:
        pass
    text = unicodedata.normalize('NFKC', str(value)).strip()
    if text.lower() in {'nan', 'none', 'nat'}:
        return ''
    return text


def work_name_for_filename(value) -> str:
    return norm_text(value) or '未标注作品'


def contains_vector_hint(*values) -> bool:
    combined = ' '.join(norm_text(v) for v in values)
    return any(phrase in combined for phrase in VECTOR_HINT_PHRASES)


def safe_filename_part(value, default='unknown', max_len=48) -> str:
    text = norm_text(value) or default
    invalid_chars = set('/\\:*?"<>|')
    text = ''.join('_' if ch in invalid_chars or ord(ch) < 32 else ch for ch in text)
    text = re.sub(r'\s+', '_', text)
    text = re.sub(r'_+', '_', text).strip('._ ')
    return (text or default)[:max_len]


def parse_download_filename(path: Path) -> dict:
    parts = path.stem.split('_')
    serial_text = parts[0] if len(parts) > 0 else ''
    try:
        serial_num = int(serial_text)
    except ValueError:
        serial_num = 10**12

    return {
        'path': path,
        'serial_num': serial_num,
        'file_query_char': parts[1] if len(parts) > 1 else '',
        'file_style_name': parts[2] if len(parts) > 2 else path.parent.name,
        'file_author': parts[3] if len(parts) > 3 else '',
        'file_work_title': '_'.join(parts[4:]) if len(parts) > 4 else '',
        'filename': path.name,
    }


def row_text_values(row) -> list:
    return [row.get(col, '') for col in TEXT_COLUMNS]


def output_filename(row) -> str:
    global_index = int(row['global_index'])
    query_char = safe_filename_part(row.get('query_char'), default='unknown_char', max_len=12)
    style_label = safe_filename_part(row.get('style_label'), default='unknown_style', max_len=12)
    author = safe_filename_part(row.get('author'), default='unknown_author', max_len=48)
    return f'{global_index:08d}_{query_char}_{style_label}_{author}.png'


## 3. Load CSV metadata and build reliable local image matching

For each bundle, the notebook reads CSV rows, scans `{bundle}/*_downloads/{style_name}/`, and matches target-style metadata rows to local files.

Matching order:

1. `query_char + style_name + author + work_title`
2. `query_char + style_name + author`
3. `query_char + style_name`
4. next unused file in that style folder order
5. `no_image_found`


In [ ]:
def make_file_indexes(file_records):
    exact = defaultdict(deque)
    char_author = defaultdict(deque)
    char_style = defaultdict(deque)
    by_style = defaultdict(deque)

    for rec in file_records:
        query_char = norm_text(rec['file_query_char'])
        style_name = norm_text(rec['file_style_name'])
        author = norm_text(rec['file_author'])
        work_title = work_name_for_filename(rec['file_work_title'])

        exact[(query_char, style_name, author, work_title)].append(rec)
        char_author[(query_char, style_name, author)].append(rec)
        char_style[(query_char, style_name)].append(rec)
        by_style[style_name].append(rec)

    return exact, char_author, char_style, by_style


def pop_unused(queue, used_paths):
    while queue:
        rec = queue.popleft()
        if rec['path'] not in used_paths:
            used_paths.add(rec['path'])
            return rec
    return None


def match_image_for_row(row, indexes, used_paths):
    exact, char_author, char_style, by_style = indexes

    query_char = norm_text(row.get('query_char'))
    style_name = norm_text(row.get('style_name'))
    author = norm_text(row.get('author'))
    work_title = work_name_for_filename(row.get('work_title'))

    attempts = [
        ('filename_query_style_author_work', exact[(query_char, style_name, author, work_title)]),
        ('fallback_query_style_author', char_author[(query_char, style_name, author)]),
        ('fallback_query_char_style', char_style[(query_char, style_name)]),
        ('fallback_style_folder_order', by_style[style_name]),
    ]

    for matched_by, queue in attempts:
        rec = pop_unused(queue, used_paths)
        if rec is not None:
            return rec, matched_by

    return None, 'no_image_found'


def read_bundle_csv(bundle_name: str, global_start: int):
    bundle_path = ROOT / bundle_name
    csv_files = sorted(bundle_path.glob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No CSV found in {bundle_path}')

    csv_path = csv_files[0]
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    df.columns = [str(col).strip().lstrip('\ufeff') for col in df.columns]

    missing_cols = [col for col in ORIGINAL_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f'{csv_path} missing columns: {missing_cols}')

    df = df[ORIGINAL_COLUMNS].copy()
    df.insert(0, 'global_index', range(global_start, global_start + len(df)))
    df['bundle'] = bundle_name
    df['style_label'] = df['style_name'].map(STYLE_MAP)
    return df


def scan_bundle_image_files(bundle_name: str):
    bundle_path = ROOT / bundle_name
    download_dirs = sorted(bundle_path.glob('*_downloads'))
    if not download_dirs:
        raise FileNotFoundError(f'No *_downloads directory found in {bundle_path}')

    download_root = download_dirs[0]
    records = []
    for style_name in STYLE_MAP:
        style_dir = download_root / style_name
        if not style_dir.exists():
            continue
        for path in style_dir.iterdir():
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(parse_download_filename(path))

    records.sort(key=lambda rec: (rec['file_style_name'], rec['serial_num'], rec['filename']))
    return records


all_rows = []
matched_rows = []
failure_rows = []
bundle_stats = []
global_start = 0

for bundle_name in BUNDLE_NAMES:
    df = read_bundle_csv(bundle_name, global_start)
    global_start += len(df)

    file_records = scan_bundle_image_files(bundle_name)
    indexes = make_file_indexes(file_records)
    used_paths = set()

    csv_rows = len(df)
    target_df = df[df['style_name'].isin(STYLE_MAP)].copy()
    non_target_df = df[~df['style_name'].isin(STYLE_MAP)].copy()

    for _, row in non_target_df.iterrows():
        failure_rows.append({
            'global_index': int(row['global_index']),
            'bundle': row['bundle'],
            'query_char': row.get('query_char'),
            'style_name': row.get('style_name'),
            'style_label': '',
            'author': row.get('author'),
            'work_title': row.get('work_title'),
            'image_url': row.get('image_url'),
            'image_path': '',
            'reason': 'non_target_style',
            'error': '',
        })

    found_paths = 0
    missing_paths = 0

    for _, row in target_df.iterrows():
        row_dict = row.to_dict()
        rec, matched_by = match_image_for_row(row_dict, indexes, used_paths)

        if rec is None:
            image_path = ''
            missing_paths += 1
        else:
            image_path = str(rec['path'])
            found_paths += 1

        row_dict['image_path'] = image_path
        row_dict['matched_by'] = matched_by
        matched_rows.append(row_dict)

    bundle_stats.append({
        'bundle': bundle_name,
        'csv_rows': csv_rows,
        'target_style_rows': len(target_df),
        'found_image_paths': found_paths,
        'missing_image_paths': missing_paths,
        'downloaded_target_files': len(file_records),
    })

meta_filtered = pd.DataFrame(matched_rows)
non_target_failures = pd.DataFrame(failure_rows)

if RUN_FULL:
    work_df = meta_filtered.copy()
else:
    work_df = meta_filtered.sample(n=min(SAMPLE_SIZE, len(meta_filtered)), random_state=RANDOM_SEED).copy()

print('Bundle matching summary:')
display(pd.DataFrame(bundle_stats))
print(f'Target rows selected for preprocessing: {len(work_df):,}')
print(f'Non-target rows recorded as failures: {len(non_target_failures):,}')


## 4. Image preprocessing functions

The output stays grayscale. The process cleans the background mildly, removes tiny isolated black components, crops the character body, pads to a square, and resizes to 96x96.


In [ ]:
class SkipImage(Exception):
    def __init__(self, reason, detail=''):
        super().__init__(reason)
        self.reason = reason
        self.detail = detail


def otsu_threshold(gray: np.ndarray) -> int:
    gray_u8 = np.asarray(gray, dtype=np.uint8)
    hist = np.bincount(gray_u8.ravel(), minlength=256).astype(np.float64)
    total = hist.sum()
    if total <= 0:
        return 127

    bins = np.arange(256, dtype=np.float64)
    weight_bg = np.cumsum(hist)
    weight_fg = total - weight_bg
    sum_bg = np.cumsum(hist * bins)
    sum_total = sum_bg[-1]
    valid = (weight_bg > 0) & (weight_fg > 0)

    between = np.zeros(256, dtype=np.float64)
    mean_bg = np.zeros(256, dtype=np.float64)
    mean_fg = np.zeros(256, dtype=np.float64)
    mean_bg[valid] = sum_bg[valid] / weight_bg[valid]
    mean_fg[valid] = (sum_total - sum_bg[valid]) / weight_fg[valid]
    between[valid] = weight_bg[valid] * weight_fg[valid] * (mean_bg[valid] - mean_fg[valid]) ** 2
    return int(np.argmax(between))


def connected_components(mask: np.ndarray):
    mask_u8 = mask.astype(np.uint8)
    if HAS_CV2:
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
        sizes = stats[:, cv2.CC_STAT_AREA]
        return labels, sizes
    if HAS_SCIPY:
        labels, _ = ndi.label(mask_u8, structure=np.ones((3, 3), dtype=np.uint8))
        sizes = np.bincount(labels.ravel())
        return labels, sizes
    return None, None


def median_reference_for_mask(gray: np.ndarray) -> np.ndarray:
    gray = np.asarray(gray, dtype=np.uint8)
    if HAS_CV2:
        return cv2.medianBlur(gray, 3)
    if HAS_SCIPY:
        return ndi.median_filter(gray, size=3).astype(np.uint8)
    return gray


def clean_background_mild(gray: np.ndarray, white_percentile: float = 88) -> np.ndarray:
    gray = np.asarray(gray, dtype=np.uint8)
    threshold = np.percentile(gray, white_percentile)
    cleaned = gray.copy()
    cleaned[cleaned >= threshold] = 255

    min_value = int(cleaned.min())
    max_value = int(cleaned.max())
    if max_value > min_value:
        if HAS_CV2:
            cleaned = cv2.normalize(cleaned, None, 0, 255, cv2.NORM_MINMAX)
        else:
            cleaned = (cleaned.astype(np.float32) - min_value) * (255.0 / (max_value - min_value))

    return np.clip(cleaned, 0, 255).astype(np.uint8)


def normalize_white_background(gray_float: np.ndarray) -> np.ndarray:
    arr = gray_float.astype(np.float32)
    if float(arr.mean()) < 127.0:
        arr = 255.0 - arr

    lo, hi = np.percentile(arr, [1.0, 99.5])
    if hi > lo + 1e-6:
        arr = (arr - lo) * (255.0 / (hi - lo))
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return clean_background_mild(arr, white_percentile=88)


def foreground_mask(gray: np.ndarray) -> np.ndarray:
    ref = median_reference_for_mask(gray)
    threshold = otsu_threshold(ref)
    return ref < threshold


def remove_small_black_components(gray: np.ndarray, min_area: int = MIN_AREA) -> np.ndarray:
    mask = foreground_mask(gray)
    labels, sizes = connected_components(mask)
    if labels is None or sizes is None or len(sizes) <= 1:
        return gray.copy()

    small_labels = np.where((np.arange(len(sizes)) != 0) & (sizes < min_area))[0]
    if len(small_labels) == 0:
        return gray.copy()

    cleaned = gray.copy()
    cleaned[np.isin(labels, small_labels)] = 255
    return cleaned


def bbox_from_mask(mask: np.ndarray):
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def validate_single_character_image(gray: np.ndarray):
    h, w = gray.shape
    aspect = max(w / max(h, 1), h / max(w, 1))
    if aspect > 2.5:
        raise SkipImage('suspicious_aspect_ratio', f'image aspect={aspect:.3f}')

    mask = foreground_mask(gray)
    fg_ratio = float(mask.mean())
    if fg_ratio < 0.005:
        raise SkipImage('foreground_too_small', f'foreground_ratio={fg_ratio:.6f}')
    if fg_ratio > 0.8:
        raise SkipImage('foreground_too_large', f'foreground_ratio={fg_ratio:.6f}')

    bbox = bbox_from_mask(mask)
    if bbox is None:
        raise SkipImage('invalid_content_bbox', 'no foreground bbox')

    x0, y0, x1, y1 = bbox
    bbox_w = x1 - x0
    bbox_h = y1 - y0
    if bbox_w <= 1 or bbox_h <= 1:
        raise SkipImage('invalid_content_bbox', f'bbox={bbox}')

    bbox_aspect = max(bbox_w / bbox_h, bbox_h / bbox_w)
    if bbox_aspect > 2.8:
        raise SkipImage('suspicious_aspect_ratio', f'bbox aspect={bbox_aspect:.3f}')

    labels, sizes = connected_components(mask)
    if sizes is not None and len(sizes) > 1:
        component_count = int(np.sum(sizes[1:] >= MIN_AREA))
        if max(h, w) >= 300 and component_count > 300:
            raise SkipImage('too_many_components', f'components={component_count}')

    return bbox


def crop_pad_resize(gray: np.ndarray, bbox, output_size: int = IMAGE_SIZE) -> Image.Image:
    h, w = gray.shape
    x0, y0, x1, y1 = bbox
    box_size = max(x1 - x0, y1 - y0)
    margin = max(2, int(round(box_size * 0.04)))

    x0 = max(0, x0 - margin)
    y0 = max(0, y0 - margin)
    x1 = min(w, x1 + margin)
    y1 = min(h, y1 + margin)
    crop = gray[y0:y1, x0:x1]

    if crop.size == 0:
        raise SkipImage('invalid_content_bbox', 'empty crop')

    ch, cw = crop.shape
    side = max(ch, cw)
    side = int(math.ceil(side * 1.16))
    side = max(side, ch, cw, 1)

    square = np.full((side, side), 255, dtype=np.uint8)
    y_offset = (side - ch) // 2
    x_offset = (side - cw) // 2
    square[y_offset:y_offset + ch, x_offset:x_offset + cw] = crop

    return Image.fromarray(square, mode='L').resize((output_size, output_size), Image.Resampling.LANCZOS)


def preprocess_image(path: Path) -> Image.Image:
    try:
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert('L')
            gray_float = np.asarray(image, dtype=np.float32)
    except (UnidentifiedImageError, OSError, FileNotFoundError) as exc:
        raise SkipImage('image_read_failed', repr(exc)) from exc

    gray = normalize_white_background(gray_float)
    gray = remove_small_black_components(gray, min_area=MIN_AREA)
    bbox = validate_single_character_image(gray)
    return crop_pad_resize(gray, bbox, output_size=IMAGE_SIZE)


## 5. Run full preprocessing

This cell writes all successful processed images and records every skipped or failed target row.


In [ ]:
success_records = []
failure_records = []

output_image_root_resolved = OUTPUT_IMAGE_ROOT.resolve()

for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc='Preprocessing target rows', mininterval=5):
    row_dict = row.to_dict()
    global_index = int(row_dict['global_index'])
    image_path_text = norm_text(row_dict.get('image_path'))
    matched_by = row_dict.get('matched_by', '')

    base_failure = {
        'global_index': global_index,
        'bundle': row_dict.get('bundle'),
        'query_char': row_dict.get('query_char'),
        'style_name': row_dict.get('style_name'),
        'style_label': row_dict.get('style_label'),
        'author': row_dict.get('author'),
        'work_title': row_dict.get('work_title'),
        'image_url': row_dict.get('image_url'),
        'image_path': image_path_text,
        'reason': '',
        'error': '',
    }

    metadata_has_hint = contains_vector_hint(*row_text_values(row_dict))
    filename_has_hint = contains_vector_hint(image_path_text)
    if metadata_has_hint or filename_has_hint:
        base_failure['reason'] = 'vector_hint_image'
        failure_records.append(base_failure)
        continue

    if not image_path_text:
        base_failure['reason'] = 'no_image_found'
        failure_records.append(base_failure)
        continue

    source_path = Path(image_path_text)
    if not source_path.exists():
        base_failure['reason'] = 'no_image_found'
        base_failure['error'] = 'matched image path does not exist'
        failure_records.append(base_failure)
        continue

    processed_path = (OUTPUT_IMAGE_ROOT / row_dict['style_label'] / output_filename(row_dict)).resolve()
    if output_image_root_resolved not in processed_path.parents:
        base_failure['reason'] = 'processing_error'
        base_failure['error'] = f'processed_path escaped output root: {processed_path}'
        failure_records.append(base_failure)
        continue

    try:
        processed = preprocess_image(source_path)
        processed.save(processed_path, format='PNG')

        success_records.append({
            'global_index': global_index,
            'bundle': row_dict.get('bundle'),
            'query_char': row_dict.get('query_char'),
            'style_code': row_dict.get('style_code'),
            'style_name': row_dict.get('style_name'),
            'style_label': row_dict.get('style_label'),
            'era': row_dict.get('era'),
            'author': row_dict.get('author'),
            'work_title': row_dict.get('work_title'),
            'detail_title': row_dict.get('detail_title'),
            'image_url': row_dict.get('image_url'),
            'thumb_url': row_dict.get('thumb_url'),
            'image_path': str(source_path),
            'processed_path': str(processed_path),
            'matched_by': matched_by,
        })
    except SkipImage as exc:
        base_failure['reason'] = exc.reason
        base_failure['error'] = exc.detail
        failure_records.append(base_failure)
    except Exception as exc:
        base_failure['reason'] = 'processing_error'
        base_failure['error'] = repr(exc) + '\n' + traceback.format_exc(limit=2)
        failure_records.append(base_failure)

success_df = pd.DataFrame(success_records)
target_failure_df = pd.DataFrame(failure_records)

failure_columns = [
    'global_index', 'bundle', 'query_char', 'style_name', 'style_label',
    'author', 'work_title', 'image_url', 'image_path', 'reason', 'error'
]
if non_target_failures.empty:
    all_failures_df = target_failure_df.reindex(columns=failure_columns)
else:
    all_failures_df = pd.concat(
        [non_target_failures.reindex(columns=failure_columns), target_failure_df.reindex(columns=failure_columns)],
        ignore_index=True,
    )

metadata_columns = [
    'global_index', 'bundle', 'query_char', 'style_code', 'style_name', 'style_label',
    'era', 'author', 'work_title', 'detail_title', 'image_url', 'thumb_url',
    'image_path', 'processed_path', 'matched_by'
]
success_df = success_df.reindex(columns=metadata_columns)

metadata_path = OUTPUT_ROOT / 'metadata.csv'
failures_path = OUTPUT_ROOT / 'preprocessing_failures.csv'
success_df.to_csv(metadata_path, index=False, encoding='utf-8-sig')
all_failures_df.to_csv(failures_path, index=False, encoding='utf-8-sig')

print(f'Metadata written to: {metadata_path}')
print(f'Failures written to: {failures_path}')


## 6. Final checks and summary statistics

This verifies that every `processed_path` in `metadata.csv` exists and prints the requested counts.


In [ ]:
processed_exists = success_df['processed_path'].map(lambda p: Path(p).exists()) if not success_df.empty else pd.Series(dtype=bool)
missing_processed_paths = success_df.loc[~processed_exists, ['global_index', 'processed_path']] if not success_df.empty else pd.DataFrame()

bundle_success_counts = success_df['bundle'].value_counts().to_dict() if not success_df.empty else {}
bundle_failure_counts = target_failure_df['bundle'].value_counts().to_dict() if not target_failure_df.empty else {}

bundle_summary = pd.DataFrame(bundle_stats)
bundle_summary['success_outputs'] = bundle_summary['bundle'].map(bundle_success_counts).fillna(0).astype(int)
bundle_summary['target_failures_or_skips'] = bundle_summary['bundle'].map(bundle_failure_counts).fillna(0).astype(int)

style_success_counts = success_df['style_label'].value_counts().reindex(STYLE_LABEL_ORDER).fillna(0).astype(int)
failure_reason_counts = all_failures_df['reason'].value_counts() if not all_failures_df.empty else pd.Series(dtype='int64')

total_target_rows = len(work_df)
total_success = len(success_df)
total_target_failures = len(target_failure_df)

print(f'Output root: {OUTPUT_ROOT}')
print(f'Processed image root: {OUTPUT_IMAGE_ROOT}')
print(f'Total target rows processed: {total_target_rows:,}')
print(f'Successful processed images: {total_success:,}')
print(f'Target failures / skips: {total_target_failures:,}')
print(f'All failure rows including non-target styles: {len(all_failures_df):,}')
print(f'metadata.csv rows: {len(success_df):,}')
print(f'preprocessing_failures.csv rows: {len(all_failures_df):,}')
print(f'All processed_path files exist: {missing_processed_paths.empty}')

print('\nPer-bundle summary:')
display(bundle_summary)

print('\nStyle success counts:')
display(style_success_counts.to_frame('success_count'))

print('\nFailure reason counts:')
display(failure_reason_counts.to_frame('count'))

if not missing_processed_paths.empty:
    print('\nMissing processed_path rows:')
    display(missing_processed_paths.head(20))

no_image_found_count = int(failure_reason_counts.get('no_image_found', 0)) if not failure_reason_counts.empty else 0
print(f'no_image_found count: {no_image_found_count:,}')
